# Baseline

In [ ]:
def baseline_model(cat_tensors, num_tensors, cardinalities, fare=None):
    n_obs = len(num_tensors[0])

    # global intercept
    alpha = pyro.sample("alpha", dist.Normal(0.0, 1.0))

    # one slope per numeric feature (2 numerics)
    beta_num = pyro.sample("beta_num", dist.Normal(torch.zeros(2), torch.ones(2)).to_event(1))

    # "flat" (non-hierarchical) categorical effects: one parameter per category level
    # This is basically like doing one-hot + linear weights, but easier with index tensors.
    cat_effect = 0.0
    for i, x_cat in enumerate(cat_tensors):
        with pyro.plate(f"cat_levels_{i}", cardinalities[i]):
            w = pyro.sample(f"w_cat_{i}", dist.Normal(0.0, 1.0))
        cat_effect = cat_effect + w[x_cat]

    # numerical part
    x_num = torch.stack(num_tensors, dim=1)  # shape (N, 2)
    mu = alpha + (x_num * beta_num).sum(-1) + cat_effect

    sigma = pyro.sample("sigma", dist.HalfCauchy(1.0))
    with pyro.plate("data", n_obs):
        pyro.sample("obs", dist.Normal(mu, sigma), obs=fare)


def run_and_eval(model_fn, cat_train, num_train, y_train, cat_test, num_test, y_test,
                 cardinalities, fare_std, epochs=2000, lr=0.05, num_samples=1000):
    pyro.clear_param_store()
    guide = AutoNormal(model_fn)
    optimizer = optim.Adam({"lr": lr})
    svi = SVI(model=model_fn, guide=guide, optim=optimizer, loss=Trace_ELBO())

    for epoch in range(epochs):
        svi.step(cat_train, num_train, cardinalities, fare=y_train)

    predictive = Predictive(model_fn, guide=guide, num_samples=num_samples, return_sites=("obs",))
    samples = predictive(cat_test, num_test, cardinalities, fare=None)
    pred_samples = samples["obs"]  # (S, N_test) on scaled space

    pred_median = torch.median(pred_samples, dim=0).values
    pred_lower = torch.quantile(pred_samples, 0.05, dim=0)
    pred_upper = torch.quantile(pred_samples, 0.95, dim=0)

    mae_scaled = torch.mean(torch.abs(pred_median - y_test)).item()
    rmse_scaled = torch.sqrt(torch.mean((pred_median - y_test) ** 2)).item()
    coverage_90 = ((y_test >= pred_lower) & (y_test <= pred_upper)).float().mean().item() * 100

    # convert scaled errors to USD like you already do
    mae_usd = mae_scaled * float(fare_std)
    rmse_usd = rmse_scaled * float(fare_std)

    return {
        "mae_scaled": mae_scaled,
        "rmse_scaled": rmse_scaled,
        "coverage_90": coverage_90,
        "mae_usd": mae_usd,
        "rmse_usd": rmse_usd,
    }


# ---- Run baseline and compare ----
baseline_metrics = run_and_eval(
    baseline_model,
    cat_tensors_train, num_tensors_train, fare_tensor_train,
    cat_tensors_test,  num_tensors_test,  fare_tensor_test,
    cardinalities=cardinalities,
    fare_std=fare_std,
    epochs=2000, lr=0.05, num_samples=1000
)

print("\nBaseline (Bayesian linear, no hierarchy)")
print(f"MAE (USD): {baseline_metrics['mae_usd']:.2f}")
print(f"RMSE (USD): {baseline_metrics['rmse_usd']:.2f}")
print(f"90% Interval Coverage: {baseline_metrics['coverage_90']:.2f}%")